In [0]:
# Databricks widget parameters
dbutils.widgets.text("volume_directory", "/Volumes/usa/osint/rss/", "Volume Directory")
dbutils.widgets.text("rss_feeds", "https://www.nibib.nih.gov/rss", "RSS Feed URLs (comma-separated)")

output_dir = dbutils.widgets.get("volume_directory")
rss_feeds_param = dbutils.widgets.get("rss_feeds")

# Parse comma-separated URLs into a list, stripping whitespace
feed_urls = [url.strip() for url in rss_feeds_param.split(",") if url.strip()]
print(f"Volume directory: {output_dir}")
print(f"RSS feeds to download ({len(feed_urls)}): {feed_urls}")

In [0]:
"""
RSS Feed Downloader
----------------------------------
Downloads RSS feeds from the provided list of URLs and
saves the XML content to the specified volume directory.

Parameters:
    volume_directory - Target volume path (e.g. /Volumes/usa/osint/rss/)
    rss_feeds - Comma-separated list of RSS feed URLs
"""

import requests
import os
import re
from urllib.parse import urlparse
from datetime import datetime

# Configuration
USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
HEADERS = {
    "User-Agent": USER_AGENT,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest": "document",
    "Sec-Fetch-Mode": "navigate",
    "Sec-Fetch-Site": "none",
    "Sec-Fetch-User": "?1",
    "Cache-Control": "max-age=0",
}

# Datestamp prefix for saved files (YYYYMMDD format)
DATE_PREFIX = datetime.now().strftime("%Y%m%d")


def sanitize_filename(url):
    """Generate a safe filename from a URL, prefixed with today's datestamp."""
    parsed = urlparse(url)
    # Use the path to create a meaningful filename
    path_parts = [p for p in parsed.path.strip("/").split("/") if p]
    if path_parts:
        name = "_".join(path_parts[-2:]) if len(path_parts) > 1 else path_parts[-1]
    else:
        name = parsed.netloc.replace(".", "_")
    # Remove unsafe characters
    name = re.sub(r'[^\w\-.]', '_', name)
    if not name.endswith(".xml"):
        name += ".xml"
    # Prefix with datestamp
    return f"{DATE_PREFIX}_{name}"


def download_feed(url, output_dir):
    """Download a single RSS feed and save it to the output directory."""
    filename = sanitize_filename(url)
    filepath = os.path.join(output_dir, filename)

    try:
        response = requests.get(url, headers=HEADERS, timeout=30)
        response.raise_for_status()

        with open(filepath, "w", encoding="utf-8") as f:
            f.write(response.text)

        print(f"  \u2713 Saved: {filename} ({len(response.text):,} bytes)")
        return True
    except requests.exceptions.RequestException as e:
        print(f"  \u2717 Failed: {filename} - {e}")
        return False


print("=" * 60)
print("RSS Feed Downloader")
print(f"Timestamp: {datetime.now().isoformat()}")
print("=" * 60)

# Ensure output directory exists
os.makedirs(output_dir, exist_ok=True)
print(f"\nOutput directory: {output_dir}")
print(f"File prefix: {DATE_PREFIX}_")

# Use the feed URLs from the parameter
rss_urls = feed_urls
print(f"\nDownloading {len(rss_urls)} feeds...\n")

# Log statement for RSS feeds being downloaded
print("RSS feeds being downloaded:")
for url in rss_urls:
    print(f"  - {url}")

success_count = 0
fail_count = 0

for url in rss_urls:
    if download_feed(url, output_dir):
        success_count += 1
    else:
        fail_count += 1

# Summary
print("\n" + "=" * 60)
print(f"Download complete: {success_count} succeeded, {fail_count} failed")
print(f"Files saved to: {output_dir}")
print("=" * 60)